SETUP THE NWB DATA

In [56]:
import sys
sys.path.insert(0, '/code/src')

import os
import glob
import numpy as np
import pandas as pd
import seaborn as sns
import pynwb
from matplotlib import pyplot as plt

sns.set_theme(context='talk', style='ticks', palette='colorblind')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['axes.titlesize'] = 'medium'
plt.rcParams['axes.titlelocation'] = 'left'
pd.set_option('display.max_columns', None)

DATA_ROOT = '/data/dynamicrouting_datacube'
SCRATCH_DIR = '/scratch/rt_analysis'
DATA_DIR = os.path.join(SCRATCH_DIR, 'data')
FIGURE_DIR = os.path.join(SCRATCH_DIR, 'figures')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)

SESSION_NUMBER = 11        # <-- change this to switch session

session_paths = sorted(glob.glob('%s/*/*.nwb.zarr' % DATA_ROOT))
session_ids = [os.path.basename(p).replace('.nwb.zarr', '') for p in session_paths]
print('%d sessions available' % len(session_ids))
for position, available_id in enumerate(session_ids):
    print('  %2d  %s' % (position, available_id))

session_id = session_ids[SESSION_NUMBER]
nwb_path = session_paths[SESSION_NUMBER]

if 'session_cache' not in globals():
    session_cache = {}
if session_id not in session_cache:
    print('\nloading %s ...' % session_id)
    session_cache[session_id] = pynwb.read_nwb(nwb_path)
else:
    print('\nusing cached %s' % session_id)

session = session_cache[session_id]
trials = session.trials[:]
print('session %d: %s, %d trials' % (SESSION_NUMBER, session_id, len(trials)))

12 sessions available
   0  662892_2023-08-24
   1  664851_2023-11-16
   2  667252_2023-09-28
   3  708016_2024-04-29
   4  712815_2024-05-22
   5  713655_2024-08-09
   6  714748_2024-06-24
   7  715710_2024-07-16
   8  741137_2024-10-10
   9  742903_2024-10-23
  10  743199_2024-12-05
  11  759434_2025-02-04

loading 759434_2025-02-04 ...


/opt/conda/lib/python3.12/site-packages/hdmf_zarr/backend.py:1699: UserWarning: Inferred dtype from zarr type. Dataset missing zarr_dtype: data   <zarr.core.Array '/processing/behavior/facemap_front_camera/data' (420648, 500) float32 read-only>
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/hdmf_zarr/backend.py:1699: UserWarning: Inferred dtype from zarr type. Dataset missing zarr_dtype: data   <zarr.core.Array '/processing/behavior/facemap_side_camera/data' (420666, 500) float32 read-only>
  warnings.warn(


session 11: 759434_2025-02-04, 545 trials


TRIAL STRUCTURE

In [57]:
# Trial structure
print('Trial structure: go %d + nogo %d + catch %d = %d, total trials %d'
      % (trials.is_go.sum(), trials.is_nogo.sum(), trials.is_catch.sum(),
         trials.is_go.sum() + trials.is_nogo.sum() + trials.is_catch.sum(),
         len(trials)))
print('hit %d + miss %d = go %d'
      % (trials.is_hit.sum(), trials.is_miss.sum(), trials.is_go.sum()))
print('false alarm %d + correct reject %d = nogo %d'
      % (trials.is_false_alarm.sum(), trials.is_correct_reject.sum(),
         trials.is_nogo.sum()))

# Responses types
n_catch_response = (trials.is_catch & trials.is_response).sum()
print('\nhit %d + false alarm %d + catch response %d = %d, is_response %d'
      % (trials.is_hit.sum(), trials.is_false_alarm.sum(), n_catch_response,
         trials.is_hit.sum() + trials.is_false_alarm.sum() + n_catch_response,
         trials.is_response.sum()))

# Response window
window_start_offset = trials.response_window_start_time - trials.stim_start_time
window_stop_offset = trials.response_window_stop_time - trials.stim_start_time
print('\nresponse window start offset: %.4f to %.4f s (median %.4f)'
      % (window_start_offset.min(), window_start_offset.max(),
         window_start_offset.median()))
print('response window stop  offset: %.4f to %.4f s (median %.4f)'
      % (window_stop_offset.min(), window_stop_offset.max(),
         window_stop_offset.median()))

# Save outcome trial structure
trials['outcome'] = np.select(
    [trials.is_hit, trials.is_miss, trials.is_false_alarm, trials.is_correct_reject],
    ['hit', 'miss', 'false alarm', 'correct reject'], default='unscored')

Trial structure: go 144 + nogo 353 + catch 48 = 545, total trials 545
hit 142 + miss 2 = go 144
false alarm 40 + correct reject 313 = nogo 353

hit 142 + false alarm 40 + catch response 1 = 183, is_response 183

response window start offset: 0.0555 to 0.0763 s (median 0.0560)
response window stop  offset: 0.9730 to 1.0103 s (median 0.9901)


SETUP BEHAVIOR DATA

In [58]:
behavior_module = session.processing['behavior']
print('behavior module contents: %s\n' % list(behavior_module.data_interfaces.keys()))

running_series = behavior_module['running_speed']
running_timestamps = running_series.timestamps[:]
running_speed = running_series.data[:]
side_camera_df = behavior_module['lp_side_camera'][:]
eye_df = behavior_module['eye_tracking'][:]
rewards_df = behavior_module['rewards'][:]

behavior module contents: ['facemap_front_camera', 'facemap_side_camera', 'running_speed', 'dlc_eye_camera', 'eye_tracking', 'licks', 'lp_front_camera', 'lp_side_camera', 'quiescent_interval_violations', 'rewards']



In [59]:
TRIAL_WINDOW = 'quiescent'
QUIESCENT_WINDOW_S = 1.5
SIDE_CAMERA_FRAME_HEIGHT = 492

def get_facial_feature(part_name, facial_features_df, frame_height=SIDE_CAMERA_FRAME_HEIGHT):
    """Vertical keypoint position, masked and interpolated."""
    confidence = facial_features_df['%s_likelihood' % part_name]
    temporal_norm = facial_features_df['%s_temporal_norm' % part_name]
    y = frame_height - facial_features_df['%s_y' % part_name].astype(float)
    y[(confidence < 0.99) | (temporal_norm > np.nanmean(temporal_norm) + 2 * np.nanstd(temporal_norm))] = np.nan
    y_centered = y - np.nanmean(y)
    y_abs_centered = np.abs(y_centered)
    y_centered[y_abs_centered > np.nanmean(y_abs_centered) + 2 * np.nanstd(y_abs_centered)] = np.nan
    return pd.Series(y_centered).ffill().bfill().to_numpy()


def get_trialwise_statistics(x, timestamps, start, stop):
    """Mean, SD and coefficient of variation within each trial window."""
    x = np.asarray(x, dtype=float)
    timestamps = np.asarray(timestamps, dtype=float)
    mean_array = np.full(len(start), np.nan)
    sd_array = np.full(len(start), np.nan)
    n_array = np.zeros(len(start), dtype=int)
    for trial_position, (window_start, window_stop) in enumerate(zip(start, stop)):
        in_window = (timestamps >= window_start) & (timestamps <= window_stop)
        n_array[trial_position] = in_window.sum()
        if n_array[trial_position] >= 2:
            mean_array[trial_position] = np.nanmean(x[in_window])
            sd_array[trial_position] = np.nanstd(x[in_window])
    with np.errstate(divide='ignore', invalid='ignore'):
        cv_array = sd_array / np.abs(mean_array)
    cv_array[~np.isfinite(cv_array)] = np.nan
    return mean_array, sd_array, cv_array, n_array


# Window of analysis
if TRIAL_WINDOW == 'quiescent':
    trial_window_start = trials.stim_start_time.values - QUIESCENT_WINDOW_S
    trial_window_stop = trials.stim_start_time.values
else:
    trial_window_start = trials.start_time.values
    trial_window_stop = trials.stop_time.values

window_duration = trial_window_stop - trial_window_start

# Features
running_speed_clean = pd.Series(running_speed).interpolate(limit_direction='both').to_numpy()

pupil_area = eye_df['pupil_area'].astype(float).to_numpy().copy()
pupil_area[eye_df['pupil_is_bad_frame'].to_numpy().astype(bool)] = np.nan
pupil_area = pd.Series(pupil_area).interpolate(limit_direction='both').to_numpy()

side_camera_timestamps = side_camera_df['timestamps'].values.astype(float)
eye_timestamps = eye_df['timestamps'].values.astype(float)

signal_sources = [('running_speed', running_speed_clean, running_timestamps), ('pupil_area', pupil_area, eye_timestamps)]
for column_name, keypoint_name in [('ear', 'ear_base_l'), ('nose', 'nose_tip'),
                                   ('jaw', 'jaw'), ('whiskers', 'whisker_pad_l_side')]:
    signal_sources.append((column_name, get_facial_feature(keypoint_name, side_camera_df), side_camera_timestamps))

# Compute mean, SD and CV per trial
for column_name, signal_array, timestamp_array in signal_sources:
    mean_array, sd_array, cv_array, n_array = get_trialwise_statistics(signal_array, timestamp_array, trial_window_start, trial_window_stop)
    trials[column_name + '_mean'] = mean_array
    trials[column_name + '_sd'] = sd_array
    trials[column_name + '_cv'] = cv_array
    trials[column_name + '_n_samples'] = n_array

BEHAVIOR_VARIABLES = [('running_speed', 'Running speed (cm/s)'),
                      ('pupil_area', 'Pupil area (px$^2$)'),
                      ('ear', 'Ear position (px)'),
                      ('nose', 'Nose position (px)'),
                      ('jaw', 'Jaw position (px)'),
                      ('whiskers', 'Whisker pad position (px)')]

summary_columns = ([c + suffix for c, _ in BEHAVIOR_VARIABLES for suffix in ['_mean', '_sd', '_cv']])
print(trials[summary_columns].describe().T[['count', 'mean', 'std', 'min', 'max']])

                    count         mean         std          min          max
running_speed_mean  545.0    35.215063   12.450815     0.427361    80.305050
running_speed_sd    545.0     5.086032    3.937153     0.864121    21.977467
running_speed_cv    545.0     0.155584    0.182366     0.018686     3.222550
pupil_area_mean     545.0  3708.381729  240.723237  2790.913791  5191.537405
pupil_area_sd       545.0   269.456925  667.451837    46.646260  6837.778861
pupil_area_cv       545.0     0.067997    0.154064     0.012293     1.413579
ear_mean            545.0    -0.000731    0.862345    -2.366458     2.445496
ear_sd              545.0     1.026897    0.266863     0.524650     2.264604
ear_cv              545.0     9.600150   50.740018     0.245304   839.111962
nose_mean           545.0    -0.240507    0.739716    -2.033625     1.268531
nose_sd             545.0     0.365252    0.158858     0.146552     1.347077
nose_cv             545.0     5.209178   36.702152     0.072064   689.991206

In [60]:
statistic_columns = [c + suffix for c, _ in BEHAVIOR_VARIABLES for suffix in ['_mean', '_sd', '_cv', '_n_samples']]
trial_export_columns = (['trial_index', 'block_index', 'rewarded_modality',
                         'stim_name', 'outcome', 'reaction_time', 'is_response',
                         'is_hit', 'is_miss', 'is_false_alarm', 'is_correct_reject',
                         'is_go', 'is_nogo', 'is_catch', 'is_rewarded',
                         'consecutive_rewarded', 'consecutive_unrewarded']
                        + statistic_columns)

trial_export = trials[[c for c in trial_export_columns if c in trials]].copy()
trial_export.insert(0, 'session_id', session_id)
trial_export.insert(1, 'trial_position', np.arange(len(trial_export)))
trial_export['trial_window'] = TRIAL_WINDOW

trial_path = os.path.join(DATA_DIR, '%s__behavior_trialwise.csv' % session_id)
trial_export.to_csv(trial_path, index=False)
print('saved %s (%d rows, %d columns)' % (trial_path, len(trial_export), trial_export.shape[1]))

# session level: the distribution of each trialwise statistic
session_row = {'session_id': session_id, 'trial_window': TRIAL_WINDOW, 'n_trials': len(trials)}
for column_name, _ in BEHAVIOR_VARIABLES:
    trial_means = trials[column_name + '_mean']
    session_row['%s_mean' % column_name] = trial_means.mean()
    session_row['%s_sd' % column_name] = trial_means.std()
    session_row['%s_cv' % column_name] = trial_means.std() / abs(trial_means.mean())
    session_row['%s_mean_within_trial_sd' % column_name] = trials[column_name + '_sd'].mean()

session_path = os.path.join(DATA_DIR, '%s__behavior_session.csv' % session_id)
pd.DataFrame([session_row]).to_csv(session_path, index=False)

saved /scratch/rt_analysis/data/759434_2025-02-04__behavior_trialwise.csv (545 rows, 41 columns)
